In [1]:
!pip install --upgrade numpy
!pip install --upgrade pandas
import pandas as pd

In [2]:
from google.colab import files
uploaded = files.upload()

Saving rooster_a3_pricing.xlsx to rooster_a3_pricing (1).xlsx


In [3]:
xls = pd.ExcelFile('rooster_a3_pricing.xlsx')

In [4]:
sheet_names = xls.sheet_names
sheet_names

['customers',
 'orders',
 'products',
 'orderlines',
 'newsletters',
 'metrics',
 'pricing',
 'metadata']

We will merge datasets to segment by behaviour

In [5]:
customers_df = xls.parse("customers")
metrics_df = xls.parse("metrics")
orders_df = xls.parse("orders")
products_df = xls.parse("products")
orderlines_df = xls.parse("orderlines")

In [6]:
customers_df_head = customers_df.head()
metrics_df_head = metrics_df.head()
orders_df_head = orders_df.head()
products_df_head = products_df.head()
orderlines_df_head = orderlines_df.head()

In [7]:
customers_df_head, metrics_df_head, orders_df_head, products_df_head, orderlines_df_head

(                     customer_email customer_first_order_date  \
 0  eml_0024fa3d63@gmail.example.net                2024-11-02   
 1  eml_002df8c679@gmail.example.net                2024-12-06   
 2  eml_0034d51935@gmail.example.net                2024-08-27   
 3  eml_0036f84cc5@gmail.example.net                2024-03-16   
 4  eml_003dca5442@gmail.example.net                2024-12-18   
 
   customer_last_order_date  customer_total_orders  customer_total_spent  
 0               2025-01-02                      2                 93.77  
 1               2024-12-06                      1                 16.80  
 2               2024-08-27                      1                 18.96  
 3               2024-07-21                      2                 67.60  
 4               2025-02-27                      2                 93.20  ,
                      customer_email  order_count  total_units  total_qty  \
 0  eml_0024fa3d63@gmail.example.net            2            6          7 

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

In [9]:
orderlines_products = pd.merge(orderlines_df, products_df, on="pvr_id", how="left")

In [10]:
orderlines_full = pd.merge(orderlines_products, orders_df[["order_id", "customer_email", "discount_pc"]], on="order_id", how="left")

In [11]:
customer_behaviour = orderlines_full.groupby("customer_email").agg(
    total_orders=("order_id", "nunique"),
    total_items=("qty_minus_refund", "sum"),
    avg_pack_size=("pack_size", "mean"),
    avg_discount_pc=("discount_pc", "mean"),
    total_units=("units", "sum"),
    product_types=("product_type", pd.Series.nunique),
).reset_index()

In [12]:
customer_behaviour = pd.merge(customer_behaviour, metrics_df[[
    "customer_email", "total_cost", "repeat_customer"
]], on="customer_email", how="left")

In [13]:
spend_threshold = customer_behaviour["total_cost"].quantile(0.75)
customer_behaviour["high_value_customer"] = customer_behaviour["total_cost"] > spend_threshold

In [14]:
customer_behaviour["uses_discounts"] = customer_behaviour["avg_discount_pc"] > 0.01
customer_behaviour["buys_bundles"] = customer_behaviour["avg_pack_size"] > 1

In [15]:
features = ["total_orders", "total_items", "avg_pack_size", "avg_discount_pc",
            "total_units", "product_types", "repeat_customer", "uses_discounts", "buys_bundles"]
X = customer_behaviour[features].copy()
y = customer_behaviour["high_value_customer"].astype(int)

I used chat GPT to gert this code because the idea I had in mind wasn't working

In [16]:
X["repeat_customer"] = X["repeat_customer"].astype(int)
X["uses_discounts"] = X["uses_discounts"].astype(int)
X["buys_bundles"] = X["buys_bundles"].astype(int)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

In [20]:
y_pred = clf.predict(X_test_scaled)
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1485
           1       0.90      0.95      0.92       342

    accuracy                           0.97      1827
   macro avg       0.94      0.96      0.95      1827
weighted avg       0.97      0.97      0.97      1827

Confusion Matrix:
[[1449   36]
 [  18  324]]


In [21]:
importances = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=False)
print("\n Feature Importance:")
print(importances)


 Feature Importance:
total_units        0.345427
product_types      0.209418
total_items        0.199370
avg_pack_size      0.103251
total_orders       0.057065
avg_discount_pc    0.031528
repeat_customer    0.029934
buys_bundles       0.019133
uses_discounts     0.004874
dtype: float64


Classification (Catehorical outcome)

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

#Logistic Regression
log_clf = LogisticRegression(max_iter=1000, random_state=42)
log_clf.fit(X_train_scaled, y_train)
y_pred_log = log_clf.predict(X_test_scaled)

#Decision Tree
tree_clf = DecisionTreeClassifier(random_state=42)
tree_clf.fit(X_train_scaled, y_train)
y_pred_tree = tree_clf.predict(X_test_scaled)

In [23]:
from sklearn.metrics import classification_report

print("Logistic Regression Report:")
print(classification_report(y_test, y_pred_log))

print("Decision Tree Report:")
print(classification_report(y_test, y_pred_tree))

print("Random Forest Report:")
print(classification_report(y_test, y_pred))

Logistic Regression Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      1485
           1       0.75      0.63      0.69       342

    accuracy                           0.89      1827
   macro avg       0.83      0.79      0.81      1827
weighted avg       0.89      0.89      0.89      1827

Decision Tree Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1485
           1       0.90      0.88      0.89       342

    accuracy                           0.96      1827
   macro avg       0.94      0.93      0.93      1827
weighted avg       0.96      0.96      0.96      1827

Random Forest Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1485
           1       0.90      0.95      0.92       342

    accuracy                           0.97      1827
   macro avg       0.94      0.96      0.95      1827
we

Regression (continous outcome)

In [24]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

reg_features = ["total_items", "avg_pack_size", "product_types", "repeat_customer"]
X_reg = customer_behaviour[reg_features].copy()
y_reg = customer_behaviour["total_cost"]

# Convert booleans
X_reg["repeat_customer"] = X_reg["repeat_customer"].astype(int)

regression_df = pd.concat([X_reg, y_reg], axis=1).dropna()
X_reg = regression_df[reg_features]
y_reg = regression_df["total_cost"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
rf_reg.fit(Xr_train, yr_train)
y_pred_rf = rf_reg.predict(Xr_test)

print(" Adjusted Random Forest Regression Performance")
print(f"R²: {r2_score(yr_test, y_pred_rf):.2f}")
print(f"MAE: {mean_absolute_error(yr_test, y_pred_rf):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(yr_test, y_pred_rf)):.2f}")

 Adjusted Random Forest Regression Performance
R²: 0.92
MAE: 1.41
RMSE: 4.56


I used chat GPT to find these numbers for my Tableau presentation

In [25]:
X_test_with_email = X_test.copy()
X_test_with_email['customer_email'] = customer_behaviour.loc[X_test.index, 'customer_email']

In [26]:
regression_output = pd.DataFrame({
    'customer_email': X_test_with_email['customer_email'].values,
    'actual_total_cost': y_test.values,
    'predicted_total_cost': y_pred
})

In [27]:
regression_output.to_csv("predicted_vs_actual.csv", index=False)

In [28]:
from google.colab import files
files.download("predicted_vs_actual.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>